# Week 26, Day 3: Review III: Fourier analysis and optimisation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w026/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: rhythms in a noisy field potential

A 4 s field potential sampled at 1,000 Hz contains an 8 Hz theta rhythm, a weaker 40 Hz gamma rhythm and noise. Compute its power spectrum with np.fft.rfft, find the frequencies of the two largest peaks (`peaks`), then low-pass filter by zeroing all components above 20 Hz and transforming back. Report the fraction of theta and gamma power left (`theta_left`, `gamma_left`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fs, T = 1000, 4.0
t = np.arange(0, T, 1 / fs)
r1 = np.random.default_rng(11)
x = 2.0 * np.sin(2 * np.pi * 8 * t) + 0.8 * np.sin(2 * np.pi * 40 * t + 1.0) + 1.5 * r1.normal(size=t.size)
X = np.fft.rfft(x); f = np.fft.rfftfreq(t.size, 1 / fs)
power = np.abs(X) ** 2
peaks = ...                                      # frequencies of the two largest values of power
Xf = X.copy(); Xf[f > 20] = 0
xf = np.fft.irfft(Xf, n=t.size)
band = lambda sig, f0: np.abs(np.fft.rfft(sig))[np.argmin(np.abs(f - f0))] ** 2
theta_left = ...
gamma_left = ...
print(f"peaks at {np.sort(peaks)} Hz;  after filtering: theta {theta_left:.3f}, gamma {gamma_left:.2e} of the original power")
plt.semilogy(f[:100 * 4], power[:100 * 4]); plt.xlabel("Hz"); plt.show()

In [ ]:
import numpy as np
def test_peaks():
    "peaks holds the two strongest spectral peaks, at 8 and 40 Hz"
    assert sorted(np.round(peaks).astype(int).tolist()) == [8, 40]

def test_filter():
    "after the low-pass filter, gamma power is cut by over 99% and theta power kept within 5%"
    assert gamma_left < 0.01 and abs(theta_left - 1) < 0.05

test_peaks()
test_filter()
print("All checks passed")

### Exercise: Problem 2: three optimisers on a ravine

Minimise f(w) = ½ wᵀ H w with H = diag(1, 50), starting from (1, 1). Run gradient descent with η = 2/(λmin + λmax), heavy-ball momentum with η = 4/(√λmin + √λmax)² and β = ((√κ − 1)/(√κ + 1))², and Adam (lr 0.05, β₁ 0.9, β₂ 0.999). Store in `steps` how many steps each needs to get f below 1e-12 (cap at 20,000).

In [ ]:
H = np.diag([1.0, 50.0]); lmin, lmax = 1.0, 50.0; kappa = lmax / lmin
f = lambda w: 0.5 * w @ H @ w
g = lambda w: H @ w

def run(update, cap=20000):
    w, state = np.array([1.0, 1.0]), {}
    for k in range(1, cap + 1):
        w = update(w, state, k)
        if f(w) < 1e-12: return k
    return cap

def gd(w, s, k):
    return w - 2 / (lmin + lmax) * g(w)

def momentum(w, s, k):
    eta = 4 / (np.sqrt(lmin) + np.sqrt(lmax)) ** 2
    beta = ((np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)) ** 2
    v = s.get("v", np.zeros(2))
    v = ...                                       # heavy ball: v = beta v - eta g(w)
    s["v"] = v
    return w + v

def adam(w, s, k, lr=0.05, b1=0.9, b2=0.999):
    m, u = s.get("m", np.zeros(2)), s.get("u", np.zeros(2))
    gr = g(w); m = b1 * m + (1 - b1) * gr; u = b2 * u + (1 - b2) * gr ** 2
    s["m"], s["u"] = m, u
    mh, uh = m / (1 - b1 ** k), u / (1 - b2 ** k)
    return ...

steps = {"gd": run(gd), "momentum": run(momentum), "adam": run(adam)}
print(steps)

In [ ]:
import numpy as np
def test_counts():
    "steps holds the number of steps each method needs to bring the loss below 1e-12"
    assert set(steps) == {"gd", "momentum", "adam"} and all(isinstance(v, int) for v in steps.values())

def test_order():
    "momentum needs far fewer steps than gradient descent with the best fixed rate"
    assert steps["momentum"] < steps["gd"] / 3

test_counts()
test_order()
print("All checks passed")

### Exercise: Problem 3: the most variable direction, two ways

Maximise the variance wᵀCw of a population's projection subject to ‖w‖ = 1. Lagrange multipliers give Cw = μw, so the answer is the top eigenvector: store it in `w_eig`. Then solve it again by projected gradient ascent: from a random start (local rng seed 3), repeatedly step w ← w + 0.1·2Cw and renormalise, 500 times, storing the result in `w_pga`.

In [ ]:
r3 = np.random.default_rng(12)
B = r3.normal(size=(8, 8)); C = B @ B.T / 8      # a covariance matrix
lam, V = np.linalg.eigh(C)
w_eig = ...
w = np.random.default_rng(3).normal(size=8); w /= np.linalg.norm(w)
for _ in range(500):
    w = ...                                       # gradient step on w^T C w, then renormalise
    w /= np.linalg.norm(w)
w_pga = w
print(f"top eigenvalue {lam[-1]:.4f};  variance found by ascent {w_pga @ C @ w_pga:.4f};  |cos| {abs(w_pga @ w_eig):.6f}")

In [ ]:
import numpy as np
def test_eig():
    "w_eig is the unit eigenvector of C with the largest eigenvalue (the Lagrange solution)"
    lam, V = np.linalg.eigh(C)
    assert abs(abs(w_eig @ V[:, -1]) - 1) < 1e-9

def test_pga():
    "projected gradient ascent reaches the same direction (|cos| above 0.999) and the same variance"
    assert abs(w_pga @ w_eig) > 0.999 and abs(w_pga @ C @ w_pga - w_eig @ C @ w_eig) < 1e-6

test_eig()
test_pga()
print("All checks passed")